In [1]:
import os, glob, json, time, shutil, subprocess, numpy as np, torch, torch.nn as nn
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run("pip -q install segmentation-models-pytorch==0.6.0", shell=True, check=True)
    import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
torch.backends.cudnn.benchmark = True

RUN, ARCH, ENC = "E5_consensus_fpn_r50", "fpn", "resnet50"
EPOCHS, PER_EPOCH, BS, LR, WD = 100, 16000, 32, 1e-3, 1e-4
STOP_AFTER = 0
NW, USE_DP = 4, False
OUTD = "/kaggle/working"
CLS = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]
print("smp", smp.__version__, "| torch", torch.__version__, "| gpus", torch.cuda.device_count())

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, "not found: " + name
    return hits[0]

os.makedirs("/tmp/data", exist_ok=True)
t0 = time.time()
need = ["X_train.npy", "Y_train.npy", "X_val.npy", "Y_val.npy", "meta_train.npz", "s1_norm.json",
        "class_counts_train.json", "X_train_p2.npy", "Y_train_p2.npy", "meta_train_p2.npz", "class_counts_train_p2.json",
        "X_train_p3.npy", "Y_train_p3.npy", "meta_train_p3.npz", "class_counts_train_p3.json"]
for n in need:
    shutil.copy(find(n), f"/tmp/data/{n}")
print("copied to /tmp/data in", round(time.time() - t0), "s", flush=True)

def mm(n):
    return np.load(f"/tmp/data/{n}", mmap_mode="r")

parts = [("X_train.npy", "Y_train.npy", "meta_train.npz", "class_counts_train.json"),
         ("X_train_p2.npy", "Y_train_p2.npy", "meta_train_p2.npz", "class_counts_train_p2.json"),
         ("X_train_p3.npy", "Y_train_p3.npy", "meta_train_p3.npz", "class_counts_train_p3.json")]
SRC, GIDX, counts = [], [], np.zeros(9)
for k, (xn, yn, mn, cn) in enumerate(parts):
    SRC.append((mm(xn), mm(yn)))
    valid = np.load(f"/tmp/data/{mn}", allow_pickle=True)["valid"]
    GIDX += [(k, int(j)) for j in np.where(valid)[0]]
    c = json.load(open(f"/tmp/data/{cn}"))
    if isinstance(c, dict):
        if all(str(x).isdigit() for x in c.keys()):
            c = [c[str(i)] for i in range(9)]
        elif "counts" in c:
            c = c["counts"]
        else:
            c = [v for v in c.values() if isinstance(v, list)][0]
    counts += np.array(c, dtype=np.float64)[:9]
XV, YV = mm("X_val.npy"), mm("Y_val.npy")
nrm = json.load(open("/tmp/data/s1_norm.json"))
MEAN = torch.tensor(nrm["mean"] * 3).view(1, 6, 1, 1).cuda()
STD = torch.tensor(nrm["std"] * 3).view(1, 6, 1, 1).cuda()
print("train samples:", len(GIDX), "| val samples:", len(XV))
share = counts / counts.sum()
print("class share %:", np.round(share * 100, 1))
W = 1.0 / np.sqrt(share + 1e-9)
W = torch.tensor(W / W.sum() * 9, dtype=torch.float32).cuda()
print("class weights:", np.round(W.cpu().numpy(), 2), flush=True)

class DS(Dataset):
    def __len__(self):
        return len(GIDX)
    def __getitem__(self, i):
        k, j = GIDX[i]
        X, Y = SRC[k]
        x, y = np.array(X[j]), np.array(Y[j, 0])
        if np.random.rand() < 0.5:
            x, y = x[:, :, ::-1], y[:, ::-1]
        if np.random.rand() < 0.5:
            x, y = x[:, ::-1], y[::-1]
        r = np.random.randint(4)
        if r:
            x, y = np.rot90(x, r, (1, 2)), np.rot90(y, r, (0, 1))
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))

ce = nn.CrossEntropyLoss(weight=W, ignore_index=255)
dice = smp.losses.DiceLoss("multiclass", ignore_index=255)

@torch.no_grad()
def evaluate(model):
    model.eval()
    conf = torch.zeros(81, dtype=torch.long, device="cuda")
    ce_sum, nb = 0.0, 0
    for a in range(0, len(XV), 64):
        b = min(a + 64, len(XV))
        x = (torch.from_numpy(np.asarray(XV[a:b])).cuda().float() - MEAN) / STD
        y = torch.from_numpy(np.asarray(YV[a:b, 0])).cuda().long()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        out = out.float()
        ce_sum += float(ce(out, y))
        nb += 1
        p = out.argmax(1)
        m = y != 255
        conf += torch.bincount(y[m] * 9 + p[m], minlength=81)
    c = conf.cpu().double().numpy().reshape(9, 9)
    tp, gt, pr = np.diag(c), c.sum(1), c.sum(0)
    iou = np.where(gt > 0, tp / np.maximum(gt + pr - tp, 1), np.nan)
    return ce_sum / nb, float(tp.sum() / c.sum()), float(np.nanmean(iou)), iou

torch.manual_seed(0)
np.random.seed(0)
net = smp.FPN(ENC, encoder_weights="imagenet", in_channels=6, classes=9).cuda()
print("encoder init: ImageNet | arch:", ARCH, ENC, "| params (M):", round(sum(p.numel() for p in net.parameters()) / 1e6, 1), flush=True)
model = nn.DataParallel(net) if (USE_DP and torch.cuda.device_count() > 1) else net
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")
ds = DS()
best, best_ep, best_iou, hist = -1.0, 0, None, []
for ep in range(1, EPOCHS + 1):
    model.train()
    t1 = time.time()
    idx = np.random.permutation(len(ds))[:PER_EPOCH].tolist()
    dl = DataLoader(ds, batch_size=BS, sampler=idx, num_workers=NW, drop_last=True, pin_memory=True)
    tot, nb = 0.0, 0
    for x, y in dl:
        x = (x.cuda(non_blocking=True).float() - MEAN) / STD
        y = y.cuda(non_blocking=True).long()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        out = out.float()
        loss = ce(out, y) + dice(out, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += float(loss.detach())
        nb += 1
    sched.step()
    dt = time.time() - t1
    vce, oa, mi, iou = evaluate(model)
    hist.append(dict(ep=ep, loss=tot / nb, val_ce=vce, oa=oa, miou=mi))
    print(f"ep {ep:3d} | loss {tot / nb:.3f} | val CE {vce:.3f} | OA {oa:.3f} | mIoU {mi:.3f} | {dt:.0f}s ({nb * BS / dt:.0f} samples/s)", flush=True)
    if ep == 1:
        print("estimated full run hours:", round(dt * EPOCHS / 3600, 1), flush=True)
    core = model.module if isinstance(model, nn.DataParallel) else model
    torch.save(core.state_dict(), f"{OUTD}/{RUN}_last.pt")
    if mi > best:
        best, best_ep, best_iou = mi, ep, iou
        torch.save(core.state_dict(), f"{OUTD}/{RUN}_best.pt")
    json.dump(dict(hist=hist, best_ep=best_ep, best_miou=best), open(f"{OUTD}/{RUN}_history.json", "w"))
    if STOP_AFTER and ep == STOP_AFTER:
        break
print("best epoch:", best_ep, "| val mIoU", round(best, 3))
print("per-class val IoU:", {c: round(float(v), 2) for c, v in zip(CLS, best_iou)})

smp 0.6.0 | torch 2.11.0+cu130 | gpus 2
copied to /tmp/data in 239 s
train samples: 45277 | val samples: 2460
class share %: [ 6.  37.4 15.   1.5  9.5  9.2  6.9 13.4  1.1]
class weights: [0.94 0.38 0.59 1.86 0.75 0.76 0.88 0.63 2.21]


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

encoder init: ImageNet | arch: fpn resnet50 | params (M): 26.1


/tmp/ipykernel_24/3120303909.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = (torch.from_numpy(np.asarray(XV[a:b])).cuda().float() - MEAN) / STD


ep   1 | loss 2.085 | val CE 1.521 | OA 0.533 | mIoU 0.345 | 141s (114 samples/s)
estimated full run hours: 3.9
ep   2 | loss 1.612 | val CE 1.474 | OA 0.530 | mIoU 0.301 | 130s (123 samples/s)
ep   3 | loss 1.482 | val CE 1.106 | OA 0.623 | mIoU 0.428 | 130s (123 samples/s)
ep   4 | loss 1.362 | val CE 1.192 | OA 0.655 | mIoU 0.428 | 131s (122 samples/s)
ep   5 | loss 1.308 | val CE 1.103 | OA 0.620 | mIoU 0.418 | 130s (123 samples/s)
ep   6 | loss 1.271 | val CE 1.217 | OA 0.598 | mIoU 0.407 | 130s (123 samples/s)
ep   7 | loss 1.226 | val CE 1.039 | OA 0.659 | mIoU 0.466 | 130s (123 samples/s)
ep   8 | loss 1.203 | val CE 1.150 | OA 0.654 | mIoU 0.454 | 130s (123 samples/s)
ep   9 | loss 1.188 | val CE 1.089 | OA 0.657 | mIoU 0.473 | 130s (123 samples/s)
ep  10 | loss 1.150 | val CE 0.995 | OA 0.665 | mIoU 0.469 | 130s (123 samples/s)
ep  11 | loss 1.118 | val CE 0.996 | OA 0.650 | mIoU 0.480 | 130s (123 samples/s)
ep  12 | loss 1.116 | val CE 1.023 | OA 0.689 | mIoU 0.494 | 130s (1